# 0. Notebook overview

This notebook runs **exactly one** PRIMA benchmark—LoCoMo, HotpotQA, or GoEmotions—per Kaggle session through the repository's canonical campaign/runtime boundary. It targets a Kaggle accelerator with **2× NVIDIA T4 (16 GiB each)** and defaults to Ollama model `qwen3:30b-a3b-instruct-2507-q4_K_M`.

The model must use both GPUs, remain fully GPU-resident, and stay below a configurable 13 GiB observed-usage ceiling per GPU. The notebook fails closed on CPU offload or insufficient hardware. It validates only the selected dataset, supports checkpoint/resume, and requires an explicit full-run gate. Outputs include benchmark-native artifacts plus a consolidated report, diagnostics, plots, logs, and one downloadable ZIP archive.

Run sections in order. Re-running after interruption is safe when `RESUME=True` and the effective configuration is unchanged.


# 1. Global configuration

Edit only this cell. Dataset paths are placeholders by design; replace the path for the benchmark you select. Full execution remains disabled until `RUN_FULL_BENCHMARK=True` is set deliberately.


In [ ]:
from pathlib import Path

CONFIG = {
    "REPOSITORY_URL": "https://github.com/yuvnahr/PRIMA_next.git",
    "GIT_REF": "dev",                         # branch, tag, or immutable commit
    "BENCHMARK_SELECTION": 1,                  # 1=LoCoMo, 2=HotpotQA, 3=GoEmotions
    "MODEL": "qwen3:30b-a3b-instruct-2507-q4_K_M",
    "OLLAMA_HOST": "127.0.0.1",
    "OLLAMA_PORT": 11434,
    "CONTEXT_LENGTH": 8192,
    "REDUCED_CONTEXT_LENGTH": 4096,
    "CONTEXT_BUDGET": 6144,
    "MAX_OUTPUT_TOKENS": 512,
    "SAFETY_OVERHEAD_TOKENS": 512,
    "GPU_MEMORY_CEILING_GIB": 13.0,
    "MIN_MODEL_GPU_DELTA_MIB": 128,
    "CUDA_VISIBLE_DEVICES": "0,1",
    "OLLAMA_NUM_PARALLEL": 1,
    "OLLAMA_FLASH_ATTENTION": True,
    "SEED": 13,
    "KAGGLE_WORKING": "/kaggle/working",
    "REPOSITORY_DIR": "/kaggle/working/PRIMA_next",
    "OUTPUT_ROOT": "/kaggle/working/prima_outputs",
    "DATASET_PATHS": {
        "locomo": "/kaggle/input/<LOCOMO_DATASET>/...",
        "hotpotqa": "/kaggle/input/<HOTPOTQA_DATASET>/...",
        "goemotions": "/kaggle/input/<GOEMOTIONS_DATASET>/...",
    },
    "RUN_SMOKE_TEST": True,
    "SMOKE_ITEMS": {"locomo": 3, "hotpotqa": 3, "goemotions": 8},
    "RUN_FULL_BENCHMARK": False,               # explicit safety gate
    "RESUME": True,
    "ALLOW_NON_KAGGLE_HARDWARE": False,
    "HEALTH_TIMEOUT_SECONDS": 120,
    "REQUEST_TIMEOUT_SECONDS": 180,
    "PROVIDER_RETRIES": 1,
    "RESOURCE_SAMPLE_SECONDS": 2.0,
    "BENCHMARK_PROFILE": {
        "locomo": "prima_full",
        "hotpotqa": "prima_full",
        "goemotions": "affect_only",
    },
    "BENCHMARK_VARIANT": {
        "locomo": "normal_prima_admission",
        "hotpotqa": "distractor",
        "goemotions": "bounded_prima_affect_decision",
    },
    "OPTIONAL_METRICS": {"locomo": [], "hotpotqa": [], "goemotions": []},
    "BERTSCORE_DEVICE": "cuda:0",
    "BERTSCORE_BATCH_SIZE": 8,
    "MAINTENANCE_MODE": "flush_before_question",
    "RUNTIME_CONFIG": {
        "affect_backend": "legacy",
        "affect_model_id": None,
        "affect_model_revision": None,
        "affect_device": "auto",
        "affect_batch_size": 4,
        "affect_thresholds_path": None,
        "affect_calibration_path": None,
        "affect_local_files_only": False,
        "affect_allow_fallback": False,
        "embedding_backend": "stable",
        "embedding_model": None,
        "embedding_dimensions": 64,
        "representation_mode": "semantic",
        "identity_normalization": True,
        "retrieval_candidate_pool_size": 30,
        "reranker_enabled": True,
        "reranker_backend": "lexical_fallback",
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    },
}
DATASET_PATHS = CONFIG["DATASET_PATHS"]  # compatibility alias; CONFIG is authoritative


# 2. Benchmark selector

Choose one integer. This resolves the sole dataset, mode, and output namespace used by every later section; no other benchmark dataset is opened.


In [ ]:
import json, os, re, shutil, subprocess, sys, time, hashlib, platform, urllib.error, urllib.request
from datetime import datetime, timezone

BENCHMARKS = {1: "locomo", 2: "hotpotqa", 3: "goemotions"}
selection = CONFIG["BENCHMARK_SELECTION"]
if not isinstance(selection, int) or selection not in BENCHMARKS:
    raise ValueError("BENCHMARK_SELECTION must be exactly 1 (LoCoMo), 2 (HotpotQA), or 3 (GoEmotions)")

SELECTED_BENCHMARK = BENCHMARKS[selection]
SELECTED_DATASET = Path(DATASET_PATHS[SELECTED_BENCHMARK]).expanduser()
WORKING_ROOT = Path(CONFIG["KAGGLE_WORKING"]).expanduser().resolve()
REPOSITORY_DIR = Path(CONFIG["REPOSITORY_DIR"]).expanduser().resolve()
OUTPUT_ROOT = Path(CONFIG["OUTPUT_ROOT"]).expanduser().resolve()
MODEL_SLUG = re.sub(r"[^A-Za-z0-9._-]+", "-", CONFIG["MODEL"]).strip("-._")
SESSION_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
STATE = {
    "selected_benchmark": SELECTED_BENCHMARK,
    "selected_dataset": str(SELECTED_DATASET),
    "session_stamp": SESSION_STAMP,
    "model_slug": MODEL_SLUG,
}
print(f"Selected benchmark: {SELECTED_BENCHMARK.upper()}")
print(f"Selected dataset:   {SELECTED_DATASET}")


# 3. Environment and hardware inspection

This cell records Python, OS, memory, disk, CUDA/GPU state, and any existing repository revision. It requires exactly two usable T4 GPUs unless the explicit development override is enabled.


In [ ]:
def run_checked(command, *, cwd=None, env=None, timeout=None, capture=True):
    result = subprocess.run(
        [str(part) for part in command], cwd=cwd, env=env, timeout=timeout,
        text=True, capture_output=capture, check=False,
    )
    if result.returncode != 0:
        message = (result.stderr or result.stdout or "no command output").strip()
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, command))}\n{message}")
    return result

def nvidia_rows():
    query = "index,name,memory.total,memory.free,memory.used,utilization.gpu"
    output = run_checked(
        ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"], timeout=20
    ).stdout
    rows = []
    for line in output.splitlines():
        values = [item.strip() for item in line.split(",")]
        if len(values) != 6:
            raise RuntimeError(f"Unexpected nvidia-smi row: {line}")
        rows.append({
            "index": int(values[0]), "name": values[1], "memory_total_mib": int(values[2]),
            "memory_free_mib": int(values[3]), "memory_used_mib": int(values[4]),
            "utilization_percent": int(values[5]),
        })
    return rows

def system_ram():
    try:
        import psutil
        memory = psutil.virtual_memory()
        return {"total_bytes": memory.total, "available_bytes": memory.available}
    except Exception:
        values = {}
        for line in Path("/proc/meminfo").read_text().splitlines():
            key, raw = line.split(":", 1)
            values[key] = int(raw.strip().split()[0]) * 1024
        return {"total_bytes": values.get("MemTotal"), "available_bytes": values.get("MemAvailable")}

gpus = nvidia_rows()
if not CONFIG["ALLOW_NON_KAGGLE_HARDWARE"]:
    if len(gpus) != 2 or any("T4" not in row["name"].upper() for row in gpus):
        raise RuntimeError(f"Expected exactly two NVIDIA T4 GPUs; detected: {gpus}")

disk = shutil.disk_usage(WORKING_ROOT if WORKING_ROOT.exists() else Path.cwd())
environment_metadata = {
    "captured_at": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "executable": sys.executable,
    "os": platform.platform(),
    "cuda_visible_devices": CONFIG["CUDA_VISIBLE_DEVICES"],
    "gpus": gpus,
    "ram": system_ram(),
    "disk": {"total_bytes": disk.total, "free_bytes": disk.free},
}
try:
    import torch
    environment_metadata["cuda"] = {
        "torch_available": bool(torch.cuda.is_available()), "torch_device_count": int(torch.cuda.device_count()),
        "torch_device_names": [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())],
    }
except Exception as exc:
    environment_metadata["cuda"] = {"nvidia_smi_available": True, "torch_probe_error": str(exc)}
if (REPOSITORY_DIR / ".git").is_dir():
    environment_metadata["repository"] = {
        "branch": run_checked(["git", "branch", "--show-current"], cwd=REPOSITORY_DIR).stdout.strip(),
        "commit": run_checked(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR).stdout.strip(),
    }
print(json.dumps(environment_metadata, indent=2))


# 4. Repository setup

The configured repository is cloned once, then the exact configured ref is fetched and checked out. Only core and benchmark requirements are installed; optional semantic-metric packages are added only when explicitly requested for LoCoMo.


In [ ]:
WORKING_ROOT.mkdir(parents=True, exist_ok=True)
if not (REPOSITORY_DIR / ".git").is_dir():
    run_checked(["git", "clone", CONFIG["REPOSITORY_URL"], str(REPOSITORY_DIR)], timeout=600)
else:
    run_checked(["git", "remote", "set-url", "origin", CONFIG["REPOSITORY_URL"]], cwd=REPOSITORY_DIR)

run_checked(["git", "fetch", "--depth", "1", "origin", CONFIG["GIT_REF"]], cwd=REPOSITORY_DIR, timeout=600)
run_checked(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPOSITORY_DIR)
requirements = [REPOSITORY_DIR / "requirements-core.txt", REPOSITORY_DIR / "requirements-benchmark.txt"]
install_args = [part for requirement in requirements for part in ("-r", str(requirement))]
run_checked([sys.executable, "-m", "pip", "install", *install_args], timeout=1200, capture=False)
if SELECTED_BENCHMARK == "locomo" and CONFIG["OPTIONAL_METRICS"]["locomo"]:
    optional = REPOSITORY_DIR / "requirements-semantic-metrics.txt"
    if optional.is_file():
        run_checked([sys.executable, "-m", "pip", "install", "-r", str(optional)], timeout=1200, capture=False)

repo_metadata = {
    "url": CONFIG["REPOSITORY_URL"], "requested_ref": CONFIG["GIT_REF"],
    "commit": run_checked(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR).stdout.strip(),
    "branch": run_checked(["git", "branch", "--show-current"], cwd=REPOSITORY_DIR).stdout.strip() or "DETACHED",
    "status": run_checked(["git", "status", "--short"], cwd=REPOSITORY_DIR).stdout.splitlines(),
    "python": sys.version,
    "pip": run_checked([sys.executable, "-m", "pip", "--version"], timeout=30).stdout.strip(),
}
STATE["repository"] = repo_metadata
print(json.dumps(repo_metadata, indent=2))


# 5. Ollama setup

Ollama is installed if absent, stale servers are stopped, and one server is started with both GPUs visible, parallelism 1, the configured context length, and flash attention. Health failures print the preserved server log. Only the selected model is pulled.


In [ ]:
import signal, tempfile

OLLAMA_ENDPOINT = f"http://{CONFIG['OLLAMA_HOST']}:{CONFIG['OLLAMA_PORT']}"
OLLAMA_LOG = WORKING_ROOT / "ollama-server.log"
OLLAMA_PROCESS = None

def url_json(path, payload=None, timeout=10):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        OLLAMA_ENDPOINT + path, data=body,
        headers={"Content-Type": "application/json"} if body is not None else {},
        method="POST" if body is not None else "GET",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def install_ollama():
    if shutil.which("ollama"):
        return
    installer_url = "https://ollama.com/install.sh"
    with urllib.request.urlopen(installer_url, timeout=30) as response:
        script = response.read()
    with tempfile.NamedTemporaryFile("wb", suffix=".sh", delete=False) as stream:
        stream.write(script)
        installer = Path(stream.name)
    try:
        run_checked(["bash", str(installer)], timeout=300, capture=False)
    finally:
        installer.unlink(missing_ok=True)

def stop_ollama():
    global OLLAMA_PROCESS
    subprocess.run(["ollama", "stop", CONFIG["MODEL"]], text=True, capture_output=True, timeout=30, check=False)
    if OLLAMA_PROCESS and OLLAMA_PROCESS.poll() is None:
        OLLAMA_PROCESS.terminate()
        try:
            OLLAMA_PROCESS.wait(timeout=15)
        except subprocess.TimeoutExpired:
            OLLAMA_PROCESS.kill()
    subprocess.run(["pkill", "-f", "ollama serve"], text=True, capture_output=True, timeout=20, check=False)
    OLLAMA_PROCESS = None

def start_ollama(context_length):
    global OLLAMA_PROCESS
    stop_ollama()
    service_env = dict(os.environ)
    service_env.update({
        "CUDA_VISIBLE_DEVICES": CONFIG["CUDA_VISIBLE_DEVICES"],
        "OLLAMA_HOST": f"{CONFIG['OLLAMA_HOST']}:{CONFIG['OLLAMA_PORT']}",
        "OLLAMA_NUM_PARALLEL": str(CONFIG["OLLAMA_NUM_PARALLEL"]),
        "OLLAMA_CONTEXT_LENGTH": str(context_length),
        "OLLAMA_FLASH_ATTENTION": "1" if CONFIG["OLLAMA_FLASH_ATTENTION"] else "0",
    })
    log_stream = OLLAMA_LOG.open("a", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"], stdout=log_stream, stderr=subprocess.STDOUT,
        text=True, env=service_env, start_new_session=True,
    )
    deadline = time.monotonic() + CONFIG["HEALTH_TIMEOUT_SECONDS"]
    while time.monotonic() < deadline:
        if OLLAMA_PROCESS.poll() is not None:
            break
        try:
            version = url_json("/api/version", timeout=3)
            STATE["ollama_version"] = version
            STATE["ollama_context_length"] = context_length
            return version
        except Exception:
            time.sleep(1)
    tail = "\n".join(OLLAMA_LOG.read_text(errors="replace").splitlines()[-80:]) if OLLAMA_LOG.exists() else ""
    raise RuntimeError(f"Ollama failed health check. Log tail:\n{tail}")

def stream_command(command, *, env=None):
    process = subprocess.Popen(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.rstrip())
    if process.wait() != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")

install_ollama()
version = start_ollama(CONFIG["CONTEXT_LENGTH"])
client_env = dict(os.environ, OLLAMA_HOST=OLLAMA_ENDPOINT)
pull_started = time.perf_counter()
stream_command(["ollama", "pull", CONFIG["MODEL"]], env=client_env)
STATE["ollama_pull_seconds"] = time.perf_counter() - pull_started
print("Ollama version:", version)


# 6. Model GPU preflight

A tiny deterministic request loads the model. Placement is verified using both `ollama ps` and structured `/api/ps`, while `nvidia-smi` confirms observed use on both GPUs. If placement or the soft ceiling fails, the notebook unloads the model, reduces context, and retries exactly once; a second failure stops execution.


In [ ]:
def warm_model():
    before = nvidia_rows()
    started = time.perf_counter()
    result = url_json("/api/generate", {
        "model": CONFIG["MODEL"], "prompt": "Reply with exactly: READY", "stream": False,
        "options": {"temperature": 0, "seed": CONFIG["SEED"], "num_ctx": STATE["ollama_context_length"], "num_predict": 8},
        "keep_alive": "30m",
    }, timeout=CONFIG["REQUEST_TIMEOUT_SECONDS"])
    elapsed = time.perf_counter() - started
    after = nvidia_rows()
    return before, after, result, elapsed

def placement_snapshot(before, after, warmup, latency):
    ps_text = run_checked(["ollama", "ps"], env=dict(os.environ, OLLAMA_HOST=OLLAMA_ENDPOINT), timeout=20).stdout
    api_ps = url_json("/api/ps", timeout=10)
    models = api_ps.get("models", [])
    active = next((row for row in models if row.get("name") == CONFIG["MODEL"] or row.get("model") == CONFIG["MODEL"]), None)
    if active is None:
        raise RuntimeError(f"Expected model is not active: {CONFIG['MODEL']}; /api/ps={api_ps}")
    size = int(active.get("size") or 0)
    size_vram = int(active.get("size_vram") or 0)
    residency = size_vram / size if size else 0.0
    deltas = [a["memory_used_mib"] - b["memory_used_mib"] for b, a in zip(before, after)]
    context = int(active.get("context_length") or STATE["ollama_context_length"])
    full_gpu = residency >= 0.98 and "100% GPU" in ps_text.upper() and "CPU" not in ps_text.upper()
    both_gpu = len(deltas) == 2 and all(delta >= CONFIG["MIN_MODEL_GPU_DELTA_MIB"] for delta in deltas)
    within_limit = all(row["memory_used_mib"] <= CONFIG["GPU_MEMORY_CEILING_GIB"] * 1024 for row in after)
    return {
        "captured_at": datetime.now(timezone.utc).isoformat(), "endpoint": OLLAMA_ENDPOINT,
        "endpoint_health": STATE["ollama_version"], "model": CONFIG["MODEL"],
        "ollama_ps": ps_text, "api_ps": api_ps, "gpu_before": before, "gpu_after": after,
        "gpu_memory_delta_mib": deltas, "residency_ratio": residency,
        "fully_gpu_resident": full_gpu, "both_gpus_used": both_gpu, "within_soft_limit": within_limit,
        "actual_context_length": context, "warmup_latency_seconds": latency,
        "warmup_response": str(warmup.get("response", ""))[:100],
    }

preflight = None
for attempt, context in enumerate((CONFIG["CONTEXT_LENGTH"], CONFIG["REDUCED_CONTEXT_LENGTH"]), start=1):
    if attempt == 2:
        print(f"Retrying once with reduced context: {context}")
        start_ollama(context)
    before, after, warmup, latency = warm_model()
    candidate = placement_snapshot(before, after, warmup, latency)
    if candidate["fully_gpu_resident"] and candidate["both_gpus_used"] and candidate["within_soft_limit"]:
        preflight = candidate
        break
    print(json.dumps(candidate, indent=2))

if preflight is None:
    stop_ollama()
    raise RuntimeError("Model GPU preflight failed after one reduced-context retry; CPU offload or unsafe VRAM use is not permitted")

STATE["model_load_seconds"] = preflight["warmup_latency_seconds"]
STATE["preflight"] = preflight
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "model_gpu_preflight.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print(json.dumps(preflight, indent=2))


# 7. Dataset validation

Only the selected dataset is imported and validated through its repository loader. The source is never modified. A SHA-256 fingerprint, selected item count, format, and required-field validation summary are retained.


In [ ]:
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
os.chdir(REPOSITORY_DIR)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if "<" in str(SELECTED_DATASET) or not SELECTED_DATASET.is_file():
    raise FileNotFoundError(f"Replace the {SELECTED_BENCHMARK} dataset placeholder with an existing file: {SELECTED_DATASET}")

dataset_summary = {"benchmark": SELECTED_BENCHMARK, "path": str(SELECTED_DATASET), "sha256": sha256_file(SELECTED_DATASET)}
if SELECTED_BENCHMARK == "locomo":
    from benchmarks.locomo.loader import LoCoMoDataset
    conversations = list(LoCoMoDataset(SELECTED_DATASET).conversations())
    dataset_summary.update(
        format="LoCoMo JSON", conversations=len(conversations),
        items=sum(len(row.questions) for row in conversations),
        required_fields=["conversation", "questions", "timestamps", "evidence IDs"],
    )
elif SELECTED_BENCHMARK == "hotpotqa":
    from benchmarks.hotpotqa.loader import HotpotQADataset
    loader = HotpotQADataset(SELECTED_DATASET, CONFIG["BENCHMARK_VARIANT"]["hotpotqa"])
    raw_rows = loader.load()
    missing_gold = [index for index, row in enumerate(raw_rows) if "answer" not in row or "supporting_facts" not in row]
    if missing_gold:
        raise ValueError(f"HotpotQA evaluation rows require answer and supporting_facts; missing at rows {missing_gold[:10]}")
    rows = list(loader.conversations())
    dataset_summary.update(
        format="HotpotQA JSON", items=len(rows),
        required_fields=["_id", "question", "answer", "context", "supporting_facts"],
    )
else:
    if SELECTED_DATASET.name != "test.tsv":
        raise ValueError("GoEmotions campaign evaluation requires DATASET_PATHS['goemotions'] to point to test.tsv")
    from benchmarks.goemotions.dataset import load_examples, load_labels
    from benchmarks.goemotions.training.data import validate_splits
    split_summary = validate_splits(SELECTED_DATASET.parent)
    labels = load_labels(SELECTED_DATASET.parent / "emotions.txt")
    examples = load_examples(SELECTED_DATASET.parent / "test.tsv", SELECTED_DATASET.parent / "emotions.txt")
    hashes = {name: sha256_file(SELECTED_DATASET.parent / name) for name in ("train.tsv", "dev.tsv", "test.tsv", "emotions.txt")}
    dataset_summary.update(
        format="GoEmotions TSV", items=len(examples), labels=len(labels), files=hashes,
        split_validation=split_summary, required_fields=["text", "comma-separated label IDs", "example ID"],
    )

STATE["dataset_summary"] = dataset_summary
print(json.dumps(dataset_summary, indent=2, default=str))


# 8. PRIMA benchmark configuration

The effective campaign configuration uses one shared Ollama provider session, the canonical `PrimaRuntime`, one active GPU request, deterministic seed, typed runtime settings, checkpointing, and the measured context window. Context + output + safety overhead is validated before execution.


In [ ]:
def build_campaign_config(kind, *, full):
    actual_window = int(STATE["preflight"]["actual_context_length"])
    overhead = CONFIG["MAX_OUTPUT_TOKENS"] + CONFIG["SAFETY_OVERHEAD_TOKENS"]
    context_budget = min(CONFIG["CONTEXT_BUDGET"], actual_window - overhead)
    if context_budget <= 0 or context_budget + overhead > actual_window:
        raise ValueError(f"Invalid token budget: context={context_budget}, output+safety={overhead}, window={actual_window}")
    limit = 0 if full else int(CONFIG["SMOKE_ITEMS"][SELECTED_BENCHMARK])
    run_label = "full" if full else "smoke"
    run_root = OUTPUT_ROOT / f"{SELECTED_BENCHMARK}-{run_label}"
    options = {"parallel_workers": 1}
    if SELECTED_BENCHMARK == "locomo":
        options.update({
            "full_dataset": bool(full), "maintenance_mode": CONFIG["MAINTENANCE_MODE"],
            "max_conversations": 0 if full else 1,
            "bertscore_device": CONFIG["BERTSCORE_DEVICE"],
            "bertscore_batch_size": CONFIG["BERTSCORE_BATCH_SIZE"],
        })
    elif SELECTED_BENCHMARK == "hotpotqa":
        options.update({"maintenance_mode": CONFIG["MAINTENANCE_MODE"], "sampling": "sequential"})
    else:
        options.update({"split": "test", "batch_size": 1, "device": "auto", "bootstrap_samples": 1000})
    mode_id = f"{SELECTED_BENCHMARK}-{CONFIG['BENCHMARK_VARIANT'][SELECTED_BENCHMARK]}"
    payload = {
        "schema_version": "1.0", "output_root": str(run_root),
        "provider": {
            "kind": "ollama", "model": CONFIG["MODEL"], "revision": STATE["repository"]["commit"],
            "endpoint": OLLAMA_ENDPOINT, "health_path": "/api/version", "context_window": actual_window,
            "structured_output": True, "temperature": 0.0,
            "max_output_tokens": CONFIG["MAX_OUTPUT_TOKENS"],
            "safety_overhead_tokens": CONFIG["SAFETY_OVERHEAD_TOKENS"],
            "timeout_seconds": CONFIG["REQUEST_TIMEOUT_SECONDS"], "retries": CONFIG["PROVIDER_RETRIES"],
        },
        "scheduler": {
            "mode": "sequential", "max_gpu_requests": 1, "cpu_workers": 1,
            "gpu_devices": ["0", "1"], "allow_api_concurrency": False, "min_free_disk_gb": 0.1,
        },
        "failure_policy": {"continue_benchmark_failures": True, "continue_item_failures": True},
        "telemetry": {"enabled": True, "gpu": True},
        "runtime": CONFIG["RUNTIME_CONFIG"],
        "benchmarks": [{
            "id": mode_id, "benchmark": SELECTED_BENCHMARK, "dataset_path": str(SELECTED_DATASET),
            "profile": CONFIG["BENCHMARK_PROFILE"][SELECTED_BENCHMARK],
            "variant": CONFIG["BENCHMARK_VARIANT"][SELECTED_BENCHMARK],
            "seed": CONFIG["SEED"], "context_budget": context_budget, "max_items": limit,
            "repository_mode": "in_memory", "optional_metrics": CONFIG["OPTIONAL_METRICS"][SELECTED_BENCHMARK],
            "options": options,
        }],
        "comparisons": [],
    }
    config_path = OUTPUT_ROOT / f"campaign-{SELECTED_BENCHMARK}-{run_label}.json"
    config_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return payload, config_path, run_root, mode_id

effective_config, smoke_config_path, smoke_root, mode_id = build_campaign_config("smoke", full=False)
STATE.update({"effective_config": effective_config, "mode_id": mode_id})
print(json.dumps(effective_config, indent=2))


# 9. Optional smoke test

The smoke run uses a deterministic fixed prefix through the same campaign CLI. It must complete, produce checkpoint/metrics artifacts, and leave the model safely resident before a full run is permitted.


In [ ]:
import threading
from collections import Counter
from statistics import mean

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if line.strip():
            try:
                value = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Malformed JSONL {path}:{line_number}: {exc}") from exc
            if not isinstance(value, dict):
                raise ValueError(f"Expected object in {path}:{line_number}")
            rows.append(value)
    return rows

def find_checkpoint(run_root):
    matches = list(Path(run_root).glob("modes/*/**/checkpoints/records.jsonl"))
    return matches[0] if matches else None

def campaign_command(config_path, run_root):
    command = [sys.executable, "-m", "benchmarks.campaign.cli", "run", "--config", str(config_path)]
    if CONFIG["RESUME"] and (Path(run_root) / "manifest.json").exists():
        command.append("--resume")
    return command

def execute_campaign(config_path, run_root, expected):
    run_root = Path(run_root)
    run_root.parent.mkdir(parents=True, exist_ok=True)
    log_path = run_root.parent / f"{run_root.name}.log"
    samples = []
    started = time.monotonic()
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            campaign_command(config_path, run_root), cwd=REPOSITORY_DIR,
            stdout=log, stderr=subprocess.STDOUT, text=True,
            env=dict(os.environ, OLLAMA_HOST=OLLAMA_ENDPOINT),
        )
        try:
            while process.poll() is None:
                checkpoint = find_checkpoint(run_root)
                rows = read_jsonl(checkpoint) if checkpoint else []
                statuses = Counter(str(row.get("status")) for row in rows)
                gpu = nvidia_rows()
                sample = {"timestamp": datetime.now(timezone.utc).isoformat(), "elapsed_seconds": time.monotonic() - started, "gpus": gpu}
                try:
                    sample["ram"] = system_ram()
                except Exception as exc:
                    sample["ram_error"] = str(exc)
                samples.append(sample)
                completed = len(rows)
                elapsed = sample["elapsed_seconds"]
                eta = (elapsed / completed * max(expected - completed, 0)) if completed else None
                pct = 100 * completed / expected if expected else 0
                latencies = []
                tokens = 0
                for row in rows:
                    prediction = row.get("prediction") or {}
                    timing = prediction.get("timing") or {}
                    if timing.get("total_ms") is not None:
                        latencies.append(float(timing["total_ms"]))
                    usage = prediction.get("tokens") or {}
                    tokens += int(usage.get("completion_tokens") or 0)
                peak = [max((s["gpus"][i]["memory_used_mib"] for s in samples), default=0) for i in range(2)]
                try:
                    from IPython.display import clear_output
                    clear_output(wait=True)
                except Exception:
                    pass
                print(
                    f"{completed}/{expected} ({pct:.1f}%) | elapsed {elapsed:.0f}s | "
                    f"ETA {eta:.0f}s | complete={statuses.get('complete', 0)} failed={statuses.get('failed', 0)} | "
                    f"latency latest/mean={latencies[-1] if latencies else 0:.0f}/{mean(latencies) if latencies else 0:.0f} ms | "
                    f"completion_tokens={tokens} | retries=n/a-live | GPU MiB={[g['memory_used_mib'] for g in gpu]} peak={peak}"
                )
                if any(g["memory_used_mib"] > CONFIG["GPU_MEMORY_CEILING_GIB"] * 1024 for g in gpu):
                    process.send_signal(signal.SIGINT)
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.terminate()
                        try:
                            process.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            process.kill()
                    samples_path = run_root / "resource_samples.json"
                    samples_path.write_text(json.dumps(samples, indent=2), encoding="utf-8")
                    raise RuntimeError("Observed GPU memory exceeded the configured soft ceiling; campaign stopped safely")
                time.sleep(CONFIG["RESOURCE_SAMPLE_SECONDS"])
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
            print("Campaign interrupted. Re-run this cell with RESUME=True to continue from checkpoints.")
            raise
    code_value = process.wait()
    samples_path = run_root / "resource_samples.json"
    samples_path.parent.mkdir(parents=True, exist_ok=True)
    samples_path.write_text(json.dumps(samples, indent=2), encoding="utf-8")
    if code_value != 0:
        tail = "\n".join(log_path.read_text(errors="replace").splitlines()[-100:])
        raise RuntimeError(f"Campaign exited {code_value}. Log tail:\n{tail}")
    return samples, log_path, time.monotonic() - started

if CONFIG["RUN_SMOKE_TEST"]:
    smoke_expected = min(int(CONFIG["SMOKE_ITEMS"][SELECTED_BENCHMARK]), int(dataset_summary["items"]))
    smoke_samples, smoke_log, smoke_wall_seconds = execute_campaign(smoke_config_path, smoke_root, smoke_expected)
    smoke_manifest = json.loads((smoke_root / "manifest.json").read_text())
    if smoke_manifest.get("status") != "complete":
        raise RuntimeError(f"Smoke campaign is not complete: {smoke_manifest.get('status')}")
    post_warm_started = time.perf_counter()
    post_warm = url_json("/api/generate", {
        "model": CONFIG["MODEL"], "prompt": "Reply with exactly: READY", "stream": False,
        "options": {"temperature": 0, "seed": CONFIG["SEED"], "num_ctx": STATE["ollama_context_length"], "num_predict": 8},
        "keep_alive": "30m",
    }, timeout=CONFIG["REQUEST_TIMEOUT_SECONDS"])
    post_smoke = placement_snapshot(
        STATE["preflight"]["gpu_before"], nvidia_rows(), post_warm, time.perf_counter() - post_warm_started
    )
    sampled_safe = all(
        gpu["memory_used_mib"] <= CONFIG["GPU_MEMORY_CEILING_GIB"] * 1024
        for sample in smoke_samples for gpu in sample["gpus"]
    )
    if not (post_smoke["fully_gpu_resident"] and post_smoke["both_gpus_used"] and post_smoke["within_soft_limit"] and sampled_safe):
        raise RuntimeError("Ollama GPU placement became unsafe during smoke execution")
    print("Smoke campaign passed; checkpoint, metrics, and GPU placement are valid.")
else:
    print("Smoke test skipped by explicit configuration.")


# 10. Full benchmark execution

This cell runs only the selected benchmark and only when the explicit full-run gate is true. Detailed output goes to disk; the notebook shows compact checkpoint-derived progress and sampled resource usage. Interrupting sends SIGINT to the campaign, preserving question/example-level checkpoints for resume.


In [ ]:
if not CONFIG["RUN_FULL_BENCHMARK"]:
    raise RuntimeError("Full benchmark is intentionally gated. Set RUN_FULL_BENCHMARK=True in section 1, then re-run from section 8.")

full_config, full_config_path, full_root, mode_id = build_campaign_config("full", full=True)
full_expected = int(dataset_summary["items"])
resource_samples, benchmark_log, benchmark_wall_seconds = execute_campaign(full_config_path, full_root, full_expected)
STATE.update({
    "run_root": str(full_root), "config_path": str(full_config_path), "effective_config": full_config,
    "resource_samples": resource_samples, "benchmark_log": str(benchmark_log),
    "wall_seconds": benchmark_wall_seconds,
})
print(f"Complete campaign: {full_root}")


# 11. Benchmark-specific evaluation

Metrics are loaded from the repository's validated benchmark evaluator outputs. No notebook-specific answer scoring is introduced. The displayed fields cover LoCoMo memory/evidence behavior, HotpotQA answer/supporting/joint scores, or GoEmotions multilabel and paired-transform outcomes as applicable.


In [ ]:
RUN_ROOT = Path(STATE["run_root"])
campaign_manifest = json.loads((RUN_ROOT / "manifest.json").read_text())
mode_state = campaign_manifest["modes"][STATE["mode_id"]]
child_manifest_path = RUN_ROOT / mode_state["child_manifest"]
child_root = child_manifest_path.parent
child_manifest = json.loads(child_manifest_path.read_text())
child_summary = json.loads((child_root / "summary.json").read_text())

metric_candidates = {
    "locomo": [child_root / "metrics" / "metrics.json"],
    "hotpotqa": [child_root / "metrics" / "hotpot_metrics.json", child_root / "metrics" / "hotpot_summary.json"],
    "goemotions": [child_root / "metrics" / "metrics.json"],
}[SELECTED_BENCHMARK]
metrics = {}
for candidate in metric_candidates:
    if candidate.is_file():
        value = json.loads(candidate.read_text())
        metrics.update(value if isinstance(value, dict) else {})
if not metrics:
    metrics = child_summary.get("metrics", {})
if SELECTED_BENCHMARK == "locomo":
    benchmark_evaluation = {
        "normalized_em": metrics.get("exact_match"), "token_f1": metrics.get("f1"),
        "category_metrics": metrics.get("category_metrics"), "temporal_breakdown": metrics.get("temporal_breakdown"),
        "multi_session_breakdown": metrics.get("multi_session_breakdown"),
        "abstention_category_5_accuracy": metrics.get("abstention_category_5_accuracy"),
        "candidate_evidence_recall": metrics.get("candidate_evidence_recall"),
        "final_evidence_recall": metrics.get("final_evidence_recall"),
        "memory_admission_recall": metrics.get("memory_admission_recall"), "memory_growth": metrics.get("memory_growth"),
        "maintenance_completion_rate": metrics.get("maintenance_completion_rate"),
        "failure_taxonomy": metrics.get("failure_taxonomy"),
    }
elif SELECTED_BENCHMARK == "hotpotqa":
    benchmark_evaluation = {
        key: metrics.get(key) for key in ("em", "f1", "sp_prec", "sp_recall", "sp_f1", "sp_em", "joint_em", "joint_f1")
    }
    benchmark_evaluation["retrieval_recall"] = metrics.get("sp_recall")
    benchmark_evaluation["evidence_rank"] = {
        "status": "unavailable", "reason": "The authoritative runner does not serialize a gold-evidence rank; no surrogate is fabricated."
    }
else:
    benchmark_evaluation = {
        key: metrics.get(key) for key in (
            "exact_set_accuracy", "macro_precision", "macro_recall", "macro_f1", "micro_precision", "micro_recall", "micro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1", "sample_f1", "hamming_loss", "jaccard", "cardinality_error",
            "per_class", "parse_failure_rate", "empty_output_rate", "paired_outcome_groups", "headline_affect_comparison",
        )
    }
STATE.update({"child_root": str(child_root), "child_manifest": child_manifest, "metrics": metrics, "benchmark_evaluation": benchmark_evaluation})
print(json.dumps(benchmark_evaluation, indent=2))


# 12. PRIMA architecture diagnostics

Execution coverage is derived from each response's authoritative planned/executed component lists. A component absent from the task route is `NOT_APPLICABLE`; it is never mislabeled as a runtime failure. Counts distinguish retrieval calls/results, workflow/reasoning reflection, correction attempts, accepted/rejected corrections, generation attempts, abstentions, and failures.


In [ ]:
checkpoint_path = child_root / "checkpoints" / "records.jsonl"
checkpoints = read_jsonl(checkpoint_path)

def embedded_record(checkpoint):
    prediction = checkpoint.get("prediction") or {}
    failure = checkpoint.get("failure") or {}
    diagnostics = prediction.get("diagnostics") or {}
    details = failure.get("details") or {}
    key = {"locomo": "locomo_record", "hotpotqa": "hotpot_record", "goemotions": "goemotions_record"}[SELECTED_BENCHMARK]
    return diagnostics.get(key) or details.get(key) or {}

records = [embedded_record(row) for row in checkpoints]
raw_responses = [row.get("raw_runtime_response") or row.get("runtime") or {} for row in records]
runtime_diagnostics_rows = [response.get("diagnostics") or {} for response in raw_responses if isinstance(response, dict)]
if SELECTED_BENCHMARK == "goemotions":
    runtime_diagnostics_rows.extend(
        row.get("runtime", {}) for row in records if isinstance(row.get("runtime"), dict)
    )

component_aliases = {
    "input parsing": ["input_parser"], "cognitive state": ["state_manager"],
    "affect": ["affect_engine", "emotion_classifier"],
    "dense retrieval": ["dense_retrieval"], "sparse retrieval": ["sparse_retrieval"],
    "temporal retrieval": ["temporal_retrieval"], "graph retrieval": ["graph_traversal", "graph_reasoning"],
    "fusion": ["fusion"], "reranking backend": ["reranker"],
    "context compression": ["context_compressor"], "planning": ["planner"],
    "world simulation": ["world_model"], "uncertainty gate": ["uncertainty_estimator"],
    "reflection": ["reflection"], "LLM generation": ["model_executor"],
    "tool/action execution": ["tool_executor"], "memory admission": ["memory_commit"],
    "maintenance/event processing": ["maintenance_events"],
}

def component_status(names):
    planned, executed, skipped = set(), set(), set()
    for row in runtime_diagnostics_rows:
        planned.update(map(str, row.get("planned_components", [])))
        executed.update(map(str, row.get("executed_components", [])))
        skipped.update(map(str, row.get("skipped_components", [])))
    if any(name in executed for name in names):
        return "EXECUTED"
    if any(name in planned for name in names):
        return "PLANNED_NOT_EXECUTED" if not any(name in skipped for name in names) else "SKIPPED"
    return "NOT_APPLICABLE"

def sum_diag(name):
    return sum(int(row.get(name) or 0) for row in runtime_diagnostics_rows)

attempts = [attempt for row in runtime_diagnostics_rows for attempt in row.get("correction_attempts", []) if isinstance(attempt, dict)]
architecture_diagnostics = {
    "component_coverage": {label: component_status(names) for label, names in component_aliases.items()},
    "retrieval_call_count": sum_diag("retrieval_call_count"),
    "retrieval_result_count": sum_diag("retrieval_result_count"),
    "workflow_reflection_count": sum_diag("workflow_reflection_count"),
    "reasoning_reflection_count": sum_diag("reasoning_reflection_count"),
    "correction_attempts": len(attempts),
    "accepted_corrections": sum(bool(row.get("accepted")) for row in attempts),
    "rejected_corrections": sum(not bool(row.get("accepted")) for row in attempts),
    "generation_attempts": sum_diag("model_call_count") or sum(int(row.get("model_call_count") or 0) for row in records),
    "abstentions": sum(str(row.get("outcome", "")).lower() == "abstained" for row in records),
    "runtime_failures": sum(bool(row.get("execution_failed")) for row in records),
    "executed_backends": sorted({
        str(value) for row in runtime_diagnostics_rows for details in (row.get("component_details") or {}).values()
        if isinstance(details, dict) for key, value in details.items() if "backend" in str(key).lower() and value
    }),
}
maintenance_failures = {
    str(failure.get("event_id") or json.dumps(failure, sort_keys=True))
    for row in runtime_diagnostics_rows for failure in (row.get("maintenance") or {}).get("failures", [])
    if isinstance(failure, dict)
}
architecture_diagnostics["maintenance_failure_count"] = len(maintenance_failures)
(RUN_ROOT / "runtime_diagnostics.json").write_text(json.dumps(architecture_diagnostics, indent=2), encoding="utf-8")
print(json.dumps(architecture_diagnostics, indent=2))


# 13. Performance and resource metrics

Latency/token telemetry comes from checkpoints and campaign telemetry; throughput uses full wall time. GPU and RAM peaks are notebook samples—not driver-level allocator peaks—and are labeled accordingly.


In [ ]:
def percentile(values, percentile_value):
    values = sorted(float(value) for value in values)
    if not values:
        return 0.0
    position = (len(values) - 1) * percentile_value / 100
    lower, upper = int(position), min(int(position) + 1, len(values) - 1)
    fraction = position - lower
    return values[lower] * (1 - fraction) + values[upper] * fraction

latencies, prompt_tokens, completion_tokens, total_tokens = [], 0, 0, 0
for checkpoint in checkpoints:
    prediction = checkpoint.get("prediction") or {}
    timing, usage = prediction.get("timing") or {}, prediction.get("tokens") or {}
    if timing.get("total_ms") is not None:
        latencies.append(float(timing["total_ms"]))
    prompt_tokens += int(usage.get("prompt_tokens") or 0)
    completion_tokens += int(usage.get("completion_tokens") or 0)
    total_tokens += int(usage.get("total_tokens") or 0)

samples = STATE.get("resource_samples", [])
gpu_count = len(samples[0]["gpus"]) if samples else 0
peak_gpu = [max((sample["gpus"][i]["memory_used_mib"] for sample in samples), default=0) for i in range(gpu_count)]
avg_gpu = [mean(sample["gpus"][i]["utilization_percent"] for sample in samples) for i in range(gpu_count)] if samples else []
peak_ram = max((int((sample.get("ram") or {}).get("total_bytes") or 0) - int((sample.get("ram") or {}).get("available_bytes") or 0) for sample in samples), default=0)
wall = float(STATE.get("wall_seconds") or 0)
campaign_telemetry = campaign_manifest.get("telemetry", {})
session_telemetry = campaign_telemetry.get("provider_session", campaign_telemetry)
resource_metrics = {
    "total_wall_seconds": wall, "ollama_pull_seconds": STATE.get("ollama_pull_seconds"),
    "model_load_seconds": STATE.get("model_load_seconds"),
    "warmup_latency_seconds": STATE["preflight"]["warmup_latency_seconds"],
    "mean_latency_ms": mean(latencies) if latencies else 0.0,
    "p50_latency_ms": percentile(latencies, 50), "p90_latency_ms": percentile(latencies, 90),
    "p95_latency_ms": percentile(latencies, 95), "p99_latency_ms": percentile(latencies, 99),
    "throughput_items_per_second": len(checkpoints) / wall if wall else 0.0,
    "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": total_tokens,
    "retries": session_telemetry.get("retries", 0), "timeouts": session_telemetry.get("timeouts", 0),
    "peak_observed_vram_mib": peak_gpu, "vram_measurement": "sampled nvidia-smi observations; not driver-level peak allocation",
    "average_gpu_utilization_percent": avg_gpu, "peak_observed_ram_bytes": peak_ram,
    "output_disk_bytes": sum(path.stat().st_size for path in RUN_ROOT.rglob("*") if path.is_file()),
}
(RUN_ROOT / "resource_metrics.json").write_text(json.dumps(resource_metrics, indent=2), encoding="utf-8")
print(json.dumps(resource_metrics, indent=2))


# 14. Error and failure analysis

Cases are classified from benchmark-native fields and runtime diagnostics. Stable IDs and compact diagnostics are retained without embedding large prompts. Wrong-answer decisions use evaluator-provided per-item/category fields when available rather than reimplementing metrics.


In [ ]:
def case_id(row):
    return str(row.get("case_id") or row.get("sample_id") or row.get("id") or "unknown")

failure_analysis = {
    "failed_examples": [], "wrong_predictions": [], "low_confidence_outputs": [], "abstentions": [],
    "retrieval_misses": [], "reflection_triggered": [], "improved_after_reflection": [],
    "regressed_after_reflection": [], "parsing_failures": [], "timeouts_or_retries": [],
}
for row in records:
    compact = {"id": case_id(row), "failure_category": row.get("failure_category"), "latency_ms": row.get("latency_ms") or row.get("total_latency_ms")}
    if row.get("execution_failed"):
        failure_analysis["failed_examples"].append(compact)
    scores = row.get("per_item_scores") or {}
    wrong = (scores and float(scores.get("joint_em", scores.get("em", 1))) < 1) or row.get("failure_category") == "ANSWER_SCORING_FAILURE"
    if SELECTED_BENCHMARK == "goemotions" and "gold_labels" in row:
        wrong = set(row.get("gold_labels", [])) != set(row.get("predicted_labels", []))
    if wrong:
        failure_analysis["wrong_predictions"].append(compact)
    response = row.get("raw_runtime_response") or {}
    confidence = ((response.get("output_data") or {}).get("confidence") if isinstance(response, dict) else None)
    if confidence is not None and float(confidence) < 0.5:
        failure_analysis["low_confidence_outputs"].append(compact | {"confidence": confidence})
    if str(row.get("outcome", "")).lower() == "abstained":
        failure_analysis["abstentions"].append(compact)
    if row.get("failure_category") == "RETRIEVAL_MISS" or row.get("final_evidence_recall") == 0:
        failure_analysis["retrieval_misses"].append(compact)
    diagnostics = response.get("diagnostics", {}) if isinstance(response, dict) else {}
    attempts = diagnostics.get("correction_attempts", [])
    if attempts or int(row.get("reflection_interventions") or 0):
        failure_analysis["reflection_triggered"].append(compact)
    accepted = [item for item in attempts if isinstance(item, dict) and bool(item.get("accepted"))]
    improved = any(float(item.get("utility") or item.get("confidence_delta") or 0) > 0 for item in accepted)
    regressed = any(float(item.get("utility") or item.get("confidence_delta") or 0) < 0 for item in accepted)
    if SELECTED_BENCHMARK == "goemotions" and "baseline_labels" in row:
        gold, before, after = set(row.get("gold_labels", [])), set(row.get("baseline_labels", [])), set(row.get("predicted_labels", []))
        improved = before != gold and after == gold
        regressed = before == gold and after != gold
    if improved:
        failure_analysis["improved_after_reflection"].append(compact | {"basis": "accepted positive-utility correction or paired label correction"})
    if regressed:
        failure_analysis["regressed_after_reflection"].append(compact | {"basis": "accepted negative-utility correction or paired label regression"})
    if row.get("parse_error") or row.get("failure_category") in {"ANSWER_PARSE_FAILURE", "MALFORMED_OUTPUT"}:
        failure_analysis["parsing_failures"].append(compact)
    error_text = " ".join(str(row.get(key) or "") for key in ("error", "runtime_error", "ingestion_error")).lower()
    if "timeout" in error_text or "rate limit" in error_text or int(row.get("retries") or 0):
        failure_analysis["timeouts_or_retries"].append(compact)

(RUN_ROOT / "failure_analysis.json").write_text(json.dumps(failure_analysis, indent=2), encoding="utf-8")
print(json.dumps({key: len(value) for key, value in failure_analysis.items()}, indent=2))


# 15. Human-readable report

This cell creates the requested stable top-level artifact names and a consolidated Markdown report. Any interpretation is bounded to the selected mode; no baseline-improvement claim is made without a valid paired campaign.


In [ ]:
import shutil

prediction_source = child_root / "predictions" / "records.jsonl"
failure_source = child_root / "failures" / "records.jsonl"
if prediction_source.is_file():
    shutil.copy2(prediction_source, RUN_ROOT / "predictions.jsonl")
else:
    (RUN_ROOT / "predictions.jsonl").write_text("", encoding="utf-8")
if failure_source.is_file():
    shutil.copy2(failure_source, RUN_ROOT / "failures.jsonl")
else:
    (RUN_ROOT / "failures.jsonl").write_text("", encoding="utf-8")
(RUN_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

benchmark_summary = {
    "schema_version": "1.0", "status": campaign_manifest.get("status"),
    "benchmark": SELECTED_BENCHMARK, "model": CONFIG["MODEL"], "dataset": dataset_summary,
    "profile": CONFIG["BENCHMARK_PROFILE"][SELECTED_BENCHMARK],
    "variant": CONFIG["BENCHMARK_VARIANT"][SELECTED_BENCHMARK],
    "selected_items": len(child_manifest.get("selected_ids", [])),
    "checkpoint_items": len(checkpoints), "metrics": metrics, "benchmark_evaluation": benchmark_evaluation,
}
(RUN_ROOT / "benchmark_summary.json").write_text(json.dumps(benchmark_summary, indent=2, default=str), encoding="utf-8")
(RUN_ROOT / "environment_metadata.json").write_text(json.dumps(environment_metadata | {"repository": STATE["repository"]}, indent=2), encoding="utf-8")
(RUN_ROOT / "effective_config.json").write_text(json.dumps(STATE["effective_config"], indent=2), encoding="utf-8")
logs_dir = RUN_ROOT / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)
for source, name in ((Path(STATE["benchmark_log"]), "campaign.log"), (OLLAMA_LOG, "ollama-server.log")):
    if source.is_file():
        shutil.copy2(source, logs_dir / name)
preflight_source = OUTPUT_ROOT / "model_gpu_preflight.json"
if preflight_source.is_file():
    shutil.copy2(preflight_source, RUN_ROOT / "model_gpu_preflight.json")

def markdown_json(value):
    return "```json\n" + json.dumps(value, indent=2, default=str)[:30000] + "\n```"

report = f"""# PRIMA {SELECTED_BENCHMARK} benchmark report

Status: **{campaign_manifest.get('status', 'unknown')}**  
Generated: {datetime.now(timezone.utc).isoformat()}

## 1. Environment
{markdown_json(environment_metadata | {'repository': STATE['repository']})}

## 2. Model
Ollama model `{CONFIG['MODEL']}`; measured context `{STATE['preflight']['actual_context_length']}`; fully GPU resident: `{STATE['preflight']['fully_gpu_resident']}`.

## 3. Dataset
{markdown_json(dataset_summary)}

## 4. PRIMA configuration
Profile `{CONFIG['BENCHMARK_PROFILE'][SELECTED_BENCHMARK]}`, variant `{CONFIG['BENCHMARK_VARIANT'][SELECTED_BENCHMARK]}`.
{markdown_json(STATE['effective_config'])}

## 5. Headline metrics
{markdown_json(benchmark_evaluation)}

## 6. Per-category / per-label metrics
See the benchmark-native nested metric objects above and `metrics.json`.

## 7. Architecture execution coverage
{markdown_json(architecture_diagnostics['component_coverage'])}

## 8. Retrieval and reflection diagnostics
{markdown_json({key: value for key, value in architecture_diagnostics.items() if key != 'component_coverage'})}

## 9. Failures
{markdown_json({key: len(value) for key, value in failure_analysis.items()})}

## 10. Resource usage
{markdown_json(resource_metrics)}

## 11. Interpretation
These results characterize one selected PRIMA benchmark mode on the stated dataset/model/configuration. No improvement over a baseline is claimed because this notebook executes exactly one mode per session and no paired comparison was requested.

## 12. Limitations
VRAM and utilization peaks are periodic `nvidia-smi` samples, not driver allocator peaks. Results depend on the supplied Kaggle dataset snapshot, repository revision, Ollama build, and model artifact. Optional semantic metrics are omitted unless explicitly configured.
"""
(RUN_ROOT / "benchmark_report.md").write_text(report, encoding="utf-8")
print(report[:6000])


# 16. Result visualization

Plots are derived from stored metrics/checkpoints/resource samples and saved under `plots/`. The benchmark-specific panel adapts to LoCoMo categories, Hotpot answer/supporting/joint metrics, or GoEmotions per-label F1.


In [ ]:
import matplotlib.pyplot as plt

plots_dir = RUN_ROOT / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

def save_plot(name):
    plt.tight_layout()
    plt.savefig(plots_dir / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

plt.figure(figsize=(7, 4)); plt.hist(latencies, bins=min(30, max(5, len(latencies) // 5)))
plt.xlabel("Latency (ms)"); plt.ylabel("Items"); plt.title("Per-item latency")
save_plot("latency_distribution.png")

if samples:
    plt.figure(figsize=(8, 4))
    for index in range(gpu_count):
        plt.plot([s["elapsed_seconds"] for s in samples], [s["gpus"][index]["memory_used_mib"] for s in samples], label=f"GPU {index}")
    plt.axhline(CONFIG["GPU_MEMORY_CEILING_GIB"] * 1024, color="red", linestyle="--", label="soft ceiling")
    plt.xlabel("Elapsed (s)"); plt.ylabel("Observed VRAM (MiB)"); plt.title("Sampled GPU memory"); plt.legend()
    save_plot("gpu_memory.png")

failure_counts = {key: len(value) for key, value in failure_analysis.items() if value}
plt.figure(figsize=(9, 4)); plt.bar(failure_counts.keys() or ["none"], failure_counts.values() or [0])
plt.xticks(rotation=40, ha="right"); plt.ylabel("Cases"); plt.title("Failure and analysis categories")
save_plot("failure_categories.png")

if SELECTED_BENCHMARK == "goemotions":
    per_class = metrics.get("per_class", metrics.get("per_label", {}))
    labels_plot = list(per_class) if isinstance(per_class, dict) else []
    values = [float(per_class[label].get("f1", 0)) for label in labels_plot]
    plt.figure(figsize=(12, 5)); plt.bar(labels_plot, values); plt.xticks(rotation=70, ha="right"); plt.ylabel("F1"); plt.title("Per-label F1")
elif SELECTED_BENCHMARK == "locomo":
    categories = metrics.get("category_metrics", metrics.get("category_wise", {}))
    labels_plot = list(categories) if isinstance(categories, dict) else []
    values = [float(categories[label].get("f1", categories[label].get("token_f1", 0))) for label in labels_plot]
    plt.figure(figsize=(8, 4)); plt.bar(labels_plot, values); plt.ylabel("F1"); plt.title("LoCoMo category F1")
else:
    keys = [key for key in ("em", "f1", "sp_em", "sp_f1", "joint_em", "joint_f1") if key in metrics]
    plt.figure(figsize=(8, 4)); plt.bar(keys, [float(metrics[key]) for key in keys]); plt.ylabel("Score"); plt.title("HotpotQA answer / supporting / joint metrics")
save_plot("benchmark_metrics.png")


# 17. Artifact validation

Validation checks campaign and child status, exact selected/checkpoint ID reconciliation, duplicates, required metric keys, required files, and common credential patterns. Any failure marks packaging `partial` and is listed explicitly.


In [ ]:
required_metrics = {
    "locomo": ["exact_match", "f1", "failure_taxonomy"],
    "hotpotqa": ["em", "f1", "sp_f1", "joint_f1"],
    "goemotions": ["exact_set_accuracy", "macro_f1", "micro_f1", "weighted_f1", "sample_f1", "hamming_loss", "jaccard"],
}[SELECTED_BENCHMARK]
required_files = [
    "manifest.json", "benchmark_report.md", "benchmark_summary.json", "metrics.json",
    "runtime_diagnostics.json", "resource_metrics.json", "failures.jsonl", "predictions.jsonl",
    "effective_config.json", "environment_metadata.json",
]
validation_errors = []
if campaign_manifest.get("status") != "complete":
    validation_errors.append(f"campaign status is {campaign_manifest.get('status')!r}")
if child_manifest.get("status") != "complete":
    validation_errors.append(f"child status is {child_manifest.get('status')!r}")
selected_ids = list(map(str, child_manifest.get("selected_ids", [])))
checkpoint_ids = [str(row.get("case_id")) for row in checkpoints]
if len(checkpoint_ids) != len(set(checkpoint_ids)):
    validation_errors.append("duplicate checkpoint case IDs")
if set(checkpoint_ids) != set(selected_ids):
    validation_errors.append(f"selected/checkpoint ID mismatch: selected={len(selected_ids)} checkpoints={len(checkpoint_ids)}")
for key in required_metrics:
    if key not in metrics:
        validation_errors.append(f"missing required metric: {key}")
for name in required_files:
    if not (RUN_ROOT / name).is_file():
        validation_errors.append(f"missing artifact: {name}")

secret_pattern = re.compile(r"(?i)(api[_-]?key|secret|token|password)\s*[:=]\s*['\"]?[A-Za-z0-9_\-]{16,}")
for path in RUN_ROOT.rglob("*"):
    if path.is_file() and path.stat().st_size <= 10_000_000 and path.suffix.lower() in {".json", ".jsonl", ".md", ".txt", ".log", ".yaml", ".yml"}:
        if secret_pattern.search(path.read_text(encoding="utf-8", errors="ignore")):
            validation_errors.append(f"possible secret in {path.relative_to(RUN_ROOT)}")

artifact_status = "complete" if not validation_errors else "partial"
resource_metrics["output_disk_bytes"] = sum(path.stat().st_size for path in RUN_ROOT.rglob("*") if path.is_file())
(RUN_ROOT / "resource_metrics.json").write_text(json.dumps(resource_metrics, indent=2), encoding="utf-8")
validation = {"status": artifact_status, "errors": validation_errors, "selected_count": len(selected_ids), "checkpoint_count": len(checkpoint_ids)}
(RUN_ROOT / "artifact_validation.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")
STATE["artifact_status"] = artifact_status
print(json.dumps(validation, indent=2))


# 18. Package artifacts

Exactly one notebook-generated archive is created from the campaign output. It contains reports, metrics, predictions, checkpoints, logs, diagnostics, plots, manifests, effective config, and environment metadata—never datasets, repository files, environments, caches, or Ollama weights.


In [ ]:
archive_name = f"PRIMA_{SELECTED_BENCHMARK}_{MODEL_SLUG}_{SESSION_STAMP}.zip"
archive_path = WORKING_ROOT / archive_name
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_path.with_suffix("")), "zip", root_dir=RUN_ROOT)
archive_hash = sha256_file(archive_path)
STATE["archive_path"] = str(archive_path)
STATE["archive_sha256"] = archive_hash
print(f"Artifact status: {STATE['artifact_status']}")
print(f"Archive: {archive_path}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MiB")
print(f"SHA-256: {archive_hash}")
if STATE["artifact_status"] != "complete":
    print("PARTIAL archive reasons:", *validation_errors, sep="\n- ")


# 19. Download

Use the rendered link to download the single ZIP archive. The exact Kaggle working path and checksum are printed for independent verification.


In [ ]:
from IPython.display import FileLink, display

archive_path = Path(STATE["archive_path"])
print(f"Archive path: {archive_path}")
print(f"SHA-256: {STATE['archive_sha256']}")
display(FileLink(str(archive_path), result_html_prefix="Download: "))
